# 使用 Milvus 和 DeepSeek 构建 RAG（Windows 兼容版）

由于 `milvus-lite` 不支持 Windows，本 notebook 使用 numpy 进行内存向量搜索来替代。

RAG 流程完全等效：加载文档 → 生成嵌入 → 向量检索 → LLM 生成回答。

## 1. 加载环境变量

In [11]:
import os
from dotenv import load_dotenv

# .env 文件在项目根目录，需要指定路径
load_dotenv(dotenv_path=os.path.join('..', '..', '.env'))
api_key = os.getenv("DEEPSEEK_API_KEY")
if not api_key:
    raise ValueError("请在 .env 文件中设置 DEEPSEEK_API_KEY")
print(f"[✓] DeepSeek API Key 已加载 (前8位: {api_key[:8]}...)")

[✓] DeepSeek API Key 已加载 (前8位: sk-0546d...)


## 2. 准备数据 - 加载 FAQ 文档

In [12]:
from glob import glob

text_lines = []

for file_path in glob("milvus_docs/en/faq/*.md", recursive=True):
    with open(file_path, "r", encoding="utf-8") as file:
        file_text = file.read()
    text_lines += file_text.split("# ")

# 过滤掉空行和太短的内容
text_lines = [line.strip() for line in text_lines if len(line.strip()) > 10]
print(f"[✓] 加载了 {len(text_lines)} 个文本片段")
print(f"    示例: {text_lines[0][:100]}...")

[✓] 加载了 72 个文本片段
    示例: ---
id: operational_faq.md
summary: Find answers to commonly asked questions about operations in Mil...


In [13]:
len(text_lines)

72

## 3. 初始化 DeepSeek LLM 客户端

In [14]:
from openai import OpenAI

deepseek_client = OpenAI(
    api_key=api_key,
    base_url="https://api.deepseek.com/v1",
)
print("[✓] DeepSeek 客户端已初始化")

[✓] DeepSeek 客户端已初始化


## 4. 加载 Embedding 模型

In [15]:
from pymilvus import model

embedding_model = model.dense.SentenceTransformerEmbeddingFunction(
    model_name='all-MiniLM-L6-v2',
    device='cpu'
)

# 测试 embedding
test_embedding = embedding_model.encode_queries(["This is a test"])[0]
embedding_dim = len(test_embedding)
print(f"[✓] Embedding 模型已加载, 维度: {embedding_dim}")
print(f"    测试向量前5个元素: {test_embedding[:5]}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[✓] Embedding 模型已加载, 维度: 384
    测试向量前5个元素: [ 0.03061244  0.01383138 -0.02084383  0.01632793 -0.01023149]


## 5. 生成所有文档的嵌入向量

In [16]:
import numpy as np

print(f"为 {len(text_lines)} 个文本片段生成嵌入向量...")
doc_embeddings = embedding_model.encode_documents(text_lines)
doc_embeddings_np = np.array(doc_embeddings)
print(f"[✓] 嵌入矩阵形状: {doc_embeddings_np.shape}")

为 72 个文本片段生成嵌入向量...
[✓] 嵌入矩阵形状: (72, 384)


## 6. 向量检索（numpy 内积替代 Milvus）

原 notebook 使用 `MilvusClient` + IP（内积）距离度量。
由于 `milvus-lite` 不支持 Windows，这里用 numpy 实现等效的内积搜索。

In [17]:
def search_by_ip(query_text, top_k=3):
    """使用内积 (IP) 进行向量相似度搜索，等效于 Milvus 的 IP metric"""
    query_embedding = embedding_model.encode_queries([query_text])[0]
    query_np = np.array(query_embedding)

    # 计算内积 (与 Milvus IP metric 一致)
    scores = doc_embeddings_np @ query_np

    # 取 top_k
    top_indices = np.argsort(scores)[::-1][:top_k]
    results = [(text_lines[i], float(scores[i])) for i in top_indices]
    return results

In [18]:
import json

question = "How is data stored in milvus?"
print(f"检索查询: '{question}'")
print("=" * 60)

search_results = search_by_ip(question, top_k=3)
print("\n检索到的前3个相关文档片段:")
print(json.dumps(search_results, indent=4, ensure_ascii=False))

检索查询: 'How is data stored in milvus?'

检索到的前3个相关文档片段:
[
    [
        "Where does Milvus store data?\n\nMilvus deals with two types of data, inserted data and metadata. \n\nInserted data, including vector data, scalar data, and collection-specific schema, are stored in persistent storage as incremental log. Milvus supports multiple object storage backends, including [MinIO](https://min.io/), [AWS S3](https://aws.amazon.com/s3/?nc1=h_ls), [Google Cloud Storage](https://cloud.google.com/storage?hl=en#object-storage-for-companies-of-all-sizes) (GCS), [Azure Blob Storage](https://azure.microsoft.com/en-us/products/storage/blobs), [Alibaba Cloud OSS](https://www.alibabacloud.com/product/object-storage-service), and [Tencent Cloud Object Storage](https://www.tencentcloud.com/products/cos) (COS).\n\nMetadata are generated within Milvus. Each Milvus module has its own metadata that are stored in etcd.\n\n###",
        0.6488018035888672
    ],
    [
        "How does Milvus flush data?\n\nMilv

## 7. 使用 DeepSeek LLM 生成 RAG 响应

In [19]:
context = "\n".join([result[0] for result in search_results])

SYSTEM_PROMPT = """
Human: 你是一个 AI 助手。你能够从提供的上下文段落片段中找到问题的答案。
"""

USER_PROMPT = f"""
请使用以下用 <context> 标签括起来的信息片段来回答用 <question> 标签括起来的问题。最后追加原始回答的中文翻译，并用 <translated>和</translated> 标签标注。
<context>
{context}
</context>
<question>
{question}
</question>
<translated>
</translated>
"""

print("USER_PROMPT:")
print(USER_PROMPT)

USER_PROMPT:

请使用以下用 <context> 标签括起来的信息片段来回答用 <question> 标签括起来的问题。最后追加原始回答的中文翻译，并用 <translated>和</translated> 标签标注。
<context>
Where does Milvus store data?

Milvus deals with two types of data, inserted data and metadata. 

Inserted data, including vector data, scalar data, and collection-specific schema, are stored in persistent storage as incremental log. Milvus supports multiple object storage backends, including [MinIO](https://min.io/), [AWS S3](https://aws.amazon.com/s3/?nc1=h_ls), [Google Cloud Storage](https://cloud.google.com/storage?hl=en#object-storage-for-companies-of-all-sizes) (GCS), [Azure Blob Storage](https://azure.microsoft.com/en-us/products/storage/blobs), [Alibaba Cloud OSS](https://www.alibabacloud.com/product/object-storage-service), and [Tencent Cloud Object Storage](https://www.tencentcloud.com/products/cos) (COS).

Metadata are generated within Milvus. Each Milvus module has its own metadata that are stored in etcd.

###
How does Milvus flush data?

Milvus ret

In [20]:
response = deepseek_client.chat.completions.create(
    model="deepseek-chat",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT},
    ],
)

print("最终回答:")
print("-" * 60)
print(response.choices[0].message.content)
print("-" * 60)
print("\n[✓] RAG 管道执行完成!")

最终回答:
------------------------------------------------------------
Milvus stores data in two main categories: inserted data and metadata. 

Inserted data, which includes vector data, scalar data, and collection-specific schema, is stored in persistent storage as incremental logs. This data is supported by multiple object storage backends such as MinIO, AWS S3, Google Cloud Storage (GCS), Azure Blob Storage, Alibaba Cloud OSS, and Tencent Cloud Object Storage (COS).

Metadata, generated within Milvus by its various modules, is stored in etcd.

<translated>
Milvus 将数据存储分为两大类：插入的数据和元数据。

插入的数据，包括向量数据、标量数据和集合特定的模式，以增量日志的形式存储在持久化存储中。这些数据支持多种对象存储后端，如 MinIO、AWS S3、Google Cloud Storage (GCS)、Azure Blob Storage、阿里云 OSS 和腾讯云对象存储 (COS)。

元数据由 Milvus 的各个模块生成，存储在 etcd 中。
</translated>
------------------------------------------------------------

[✓] RAG 管道执行完成!
